In [ ]:
import os
import time
import urllib.request

import cv2
import numpy as np
import mediapipe as mp

# --- Camera ----------------------------------------------------------------
CAMERA_INDEX = 0
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# --- Stage 1: motion tripwire (MOG2) ---------------------------------------
MOTION_MIN_AREA = 800     # ignore MOG2 blobs smaller than this as noise
BLUR_KERNEL = (21, 21)
HISTORY = 500
VAR_THRESHOLD = 16

# --- Stage 2: person confirmation (MediaPipe Tasks ImageSegmenter) --------
SEG_MIN_AREA = 800
IDLE_FRAMES_BEFORE_SLEEP = 15  # consecutive motion-free frames before dropping segmentation

MODEL_PATH = "selfie_multiclass_256x256.tflite"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/image_segmenter/"
    "selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite"
)

def ensure_model_downloaded():
    if not os.path.exists(MODEL_PATH):
        print(f"Downloading segmentation model to {MODEL_PATH} ...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

def main():
    ensure_model_downloaded()

    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

    if not cap.isOpened():
        raise RuntimeError("Could not open camera. Check the index/connection.")

    back_sub = cv2.createBackgroundSubtractorMOG2(
        history=HISTORY, varThreshold=VAR_THRESHOLD, detectShadows=True
    )

    BaseOptions = mp.tasks.BaseOptions
    ImageSegmenter = mp.tasks.vision.ImageSegmenter
    ImageSegmenterOptions = mp.tasks.vision.ImageSegmenterOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = ImageSegmenterOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=VisionRunningMode.VIDEO,
        output_category_mask=True,
    )

    active = False       # whether stage 2 (segmentation) is currently running
    idle_count = 0        # consecutive motion-free frames while active
    video_timestamp_ms = 0  # monotonically increasing, required by VIDEO mode

    with ImageSegmenter.create_from_options(options) as segmenter:
        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    print("Failed to grab frame, exiting.")
                    break

                display = frame.copy()

                # --- Stage 1: cheap motion check & build motion mask ------
                blurred = cv2.GaussianBlur(frame, BLUR_KERNEL, 0)
                fg_mask = back_sub.apply(blurred)
                _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
                fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel, iterations=2)
                fg_mask = cv2.dilate(fg_mask, kernel, iterations=2)

                motion_contours, _ = cv2.findContours(
                    fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                )
                
                motion_now = any(
                    cv2.contourArea(c) >= MOTION_MIN_AREA for c in motion_contours
                )

                motion_mask = np.zeros(display.shape[:2], dtype=np.uint8)

                for c in motion_contours:
                    if cv2.contourArea(c) < MOTION_MIN_AREA:
                        continue
                    
                    # Draw yellow bounding box around generic motion
                    x, y, w, h = cv2.boundingRect(c)
                    cv2.rectangle(display, (x, y), (x + w, y + h), (0, 255, 255), 1)
                    
                    # Fill the contour on our black mask (painted later)
                    cv2.drawContours(motion_mask, [c], -1, 255, thickness=-1)

                if motion_now:
                    active = True

                # --- Stage 2: expensive segmentation, only while active ---
                person_detected = False
                seg_mask = None 
                
                if active:
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

                    video_timestamp_ms += 33  # monotonic counter, ~30fps step
                    result = segmenter.segment_for_video(mp_image, video_timestamp_ms)

                    category_mask = result.category_mask.numpy_view()
                    if category_mask.ndim == 3:      # some builds return (H, W, 1)
                        category_mask = category_mask[:, :, 0]
                    
                    seg_mask = category_mask != 0  # any non-background class = person
                    seg_mask_u8 = seg_mask.astype(np.uint8) * 255

                    seg_contours, _ = cv2.findContours(
                        seg_mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                    )
                    
                    for c in seg_contours:
                        if cv2.contourArea(c) < SEG_MIN_AREA:
                            continue
                        person_detected = True
                        # Thick red bounding box around the person
                        x, y, w, h = cv2.boundingRect(c)
                        cv2.rectangle(display, (x, y), (x + w, y + h), (0, 0, 255), 2)

                # --- Stage 3: Smart Overlay Application -------------------
                overlay = display.copy()

                # If MediaPipe found a person, remove those pixels from the MOG2 mask
                if person_detected and seg_mask is not None:
                    # Where seg_mask is True (person), set motion_mask to 0 (no cyan)
                    motion_mask[seg_mask] = 0 
                    
                    # Paint the person pixels Green
                    overlay[seg_mask] = (0, 255, 0)

                # Paint whatever is left in the motion mask Cyan
                overlay[motion_mask == 255] = (255, 255, 0)

                # Blend everything onto the main display exactly once
                display = cv2.addWeighted(overlay, 0.35, display, 0.65, 0)

                # --- Sleep Decision -----------------------------------------
                if motion_now or person_detected:
                    idle_count = 0
                elif active:
                    idle_count += 1
                    if idle_count > IDLE_FRAMES_BEFORE_SLEEP:
                        active = False

                # --- Status overlay -----------------------------------------
                if person_detected:
                    status, color = "PERSON", (0, 0, 255)
                elif motion_now:
                    status, color = "MOTION", (0, 255, 255)
                else:
                    status, color = "idle", (200, 200, 200)

                cv2.putText(display, status, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
                cv2.putText(
                    display, f"segmentation: {'ON' if active else 'off'}",
                    (10, display.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1
                )

                cv2.imshow("Motion trigger -> segmentation confirm", display)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

        finally:
            cap.release()
            cv2.destroyAllWindows()

if __name__ == "__main__":
    main()